# 📥 COPY INTO — Idempotent Data Ingestion into Delta Tables

## What This Notebook Covers

This notebook demonstrates the **COPY INTO** command in Databricks SQL — a simple, **idempotent** way to load data from cloud storage files (CSV, JSON, Parquet, etc.) into Delta tables.

## What We Will Achieve

1. **Set up storage** — Create a Unity Catalog managed volume and an input directory inside it
2. **Stage source files** — Copy retail CSV files from Databricks sample datasets into the volume
3. **Load data (basic)** — Use `COPY INTO` to bulk-load all CSV files into a placeholder Delta table (schema inferred automatically)
4. **Verify idempotency** — Re-run `COPY INTO` and observe that already-loaded files are skipped (0 new rows inserted)
5. **Load data (advanced)** — Create a table with an **explicit schema** and use `COPY INTO` with a `SELECT` transform to cast columns and add a custom `_insert_date` column
6. **Demonstrate incremental loading** — Add a new CSV file and re-run `COPY INTO` to load only the new data

## Why COPY INTO Matters

| Feature | Why It's Important |
| --- | --- |
| **Idempotent** | Re-running the same `COPY INTO` won't duplicate data — it tracks which files have already been loaded |
| **Schema inference** | No need to define columns upfront — `COPY INTO` can infer schema from the source files |
| **Schema control** | You can also provide an explicit table schema and use a `SELECT` to transform data during load |
| **Incremental** | New files dropped into the source path are automatically picked up on the next run |
| **Simple** | Just a few lines of SQL — no pipelines, no streaming setup needed for batch ingestion |

> 💡 Think of `COPY INTO` as a **lightweight, SQL-first alternative** to Auto Loader when you need simple, repeatable batch loads from a known file path.

In [0]:
-- CREATE NEW MANAGED VOLUMNE LANDING UNDER DEV > BRONZE
CREATE VOLUME dev.bronze.landing
COMMENT 'This is landing Manged Volume';

## Step 1 — Create a Managed Volume 🗄️

**What this cell does:** Creates a Unity Catalog **managed volume** called `dev.bronze.landing`.

**Why this is important:**
- A **volume** is a Unity Catalog object that provides a governed, filesystem-like storage area for files.
- Think of it as a **landing zone** — a place where raw files arrive before being processed into Delta tables.
- Managed volumes are stored in the catalog's managed storage location, so they're fully governed by Unity Catalog (access control, auditing, lineage).
- This follows the **medallion architecture** pattern: `bronze` (raw) → `silver` (cleaned) → `gold` (curated). Our volume lives in the **bronze** layer.

> 🔑 Key concept: **Managed volume** = Databricks controls the storage lifecycle. **External volume** = you point to your own cloud storage path.

In [0]:
%python
dbutils.fs.mkdirs('/Volumes/dev/bronze/landing/input')

True

## Step 2 — Create Input Directory 📁

**What this cell does:** Creates a sub-directory called `input` inside the volume using `dbutils.fs.mkdirs()`.

**Why this is important:**
- `COPY INTO` reads from a **file path** — not a table. Organizing files into a dedicated sub-directory keeps the landing zone clean.
- `dbutils.fs.mkdirs()` is the Python/PySpark equivalent of `mkdir -p` on a filesystem — it creates the directory (and any parent directories) if it doesn't exist.
- If you re-run this cell, it won't error — `mkdirs` is idempotent.

> 💡 Best practice: Use sub-folders like `input/`, `processed/`, `rejected/` within your volume to manage file lifecycle stages.

In [0]:
%python
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv","/Volumes/dev/bronze/landing/input")

True

## Step 3 — Stage the First CSV File 📄

**What this cell does:** Copies `2010-12-01.csv` from Databricks' built-in sample datasets into `/Volumes/dev/bronze/landing/input`.

**Why this is important:**
- This simulates a **data arrival event** — a file landing in your ingestion folder.
- The source is the **Databricks-Definitive-Guide** retail dataset — real-world e-commerce invoice data with columns like `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `UnitPrice`, `CustomerID`, `Country`.
- `dbutils.fs.cp()` performs a cloud-native copy between locations — no download/upload to local disk.

> 📦 This file contains **~5,217 rows** of retail transaction data for December 1st, 2010.

In [0]:
%python
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/dev/bronze/landing/input")

True

## Step 4 — Stage the Second CSV File 📄

**What this cell does:** Copies `2010-12-03.csv` into the same input directory.

**Why this is important:**
- This file is added **after** the first `COPY INTO` has already run (cells 6-7). This sets up the **incremental loading demo** later in the notebook.
- When we re-run `COPY INTO` in Cell 13, only this **new** file will be loaded — proving that `COPY INTO` tracks which files have already been processed.
- This is the key idempotency behavior: `COPY INTO` remembers what it has already loaded.

> 📦 This file contains **~2,202 rows** of retail transaction data for December 3rd, 2010.

In [0]:
-- CREATE PLACEHOLDER TABLE
create table dev.bronze.invoice_cp

## Step 5 — Create a Placeholder Table 🏗️

**What this cell does:** Creates an **empty Delta table** called `dev.bronze.invoice_cp` with **no defined columns**.

**Why this is important:**
- When you create a table without specifying columns, `COPY INTO` will **infer the schema** from the source files automatically.
- This is the **easiest** way to start loading data — no need to know the columns in advance.
- The tradeoff: all inferred columns will default to `STRING` type, which may not be ideal for numeric or date columns (we'll fix this in Step 11).

> ⚠️ Downside: Without explicit schema control, `Quantity`, `UnitPrice`, and `InvoiceDate` all end up as `STRING` instead of their proper types. We'll solve this later with an explicit schema + `SELECT` transform.

In [0]:
-- USE COPY INTO TO LOAD DATA INTO PLACE HOLDER TABLE
COPY INTO dev.bronze.invoice_cp
FROM '/Volumes/dev/bronze/landing/input'
FILEFORMAT = CSV
PATTERN = '*.csv'

FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5217,5217,0


## Step 6 — First COPY INTO Load 🚀

**What this cell does:** Executes `COPY INTO` to load all `*.csv` files from the volume into `dev.bronze.invoice_cp`.

**Breaking down the command:**

| Clause | Purpose |
| --- | --- |
| `COPY INTO dev.bronze.invoice_cp` | Target Delta table to load into |
| `FROM '/Volumes/dev/bronze/landing/input'` | Source file path |
| `FILEFORMAT = CSV` | Tell COPY INTO the files are CSV format |
| `PATTERN = '*.csv'` | Only pick up files matching this glob pattern |
| `FORMAT_OPTIONS ('header' = 'true')` | First row is a header row — use it for column names |
| `FORMAT_OPTIONS ('mergeSchema' = 'true')` | Allow schema evolution if files have different columns |
| `COPY_OPTIONS ('mergeSchema' = 'true')` | Merge any new columns into the target table schema |

**Result:** `5,217 rows` affected, `5,217 rows` inserted, `0 corrupt files` skipped.

> ✅ This is a **full load** — only one CSV file (`2010-12-01.csv`) was in the directory at this point, and all its rows were loaded successfully.

In [0]:
-- USE COPY INTO TO LOAD DATA INTO PLACE HOLDER TABLE
COPY INTO dev.bronze.invoice_cp
FROM '/Volumes/dev/bronze/landing/input'
FILEFORMAT = CSV
PATTERN = '*.csv'

FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


## Step 7 — Re-Run COPY INTO (Idempotency Test) 🔁

**What this cell does:** Runs the **exact same** `COPY INTO` command again.

**Why this is important:**
- The result shows `0 rows` affected, `0 rows` inserted — **nothing was loaded**!
- This proves `COPY INTO` is **idempotent**: it tracks which files have already been processed and **skips them** on subsequent runs.
- Under the hood, `COPY INTO` uses a **checkpoint** mechanism — it remembers which files were loaded based on file metadata (name, size, modification time).
- This is a critical safety feature: you can schedule `COPY INTO` to run on a cron job without worrying about duplicate data.

> 🎯 Key takeaway: `COPY INTO` = **load once, safely re-run anytime**. No duplicates, no manual deduplication needed.

In [0]:
SELECT count(1) FROM dev.bronze.invoice_cp;

count(1)
5217


## Step 9 — Verify Row Count 🔢

**What this cell does:** Runs `SELECT count(1)` to verify how many rows are in the table.

**Result:** `5,217` rows — matching exactly what was loaded in Step 6.

**Why this is important:**
- Always verify your data after loading — don't assume `COPY INTO` worked just because it returned success.
- The count confirms that the re-run in Step 7 added **zero** new rows (still 5,217, not 10,434).
- This is a simple but critical **data quality check** step in any ingestion pipeline.

In [0]:
describe extended dev.bronze.invoice_cp

col_name,data_type,comment
InvoiceNo,string,null
StockCode,string,null
Description,string,null
Quantity,string,null
InvoiceDate,string,null
UnitPrice,string,null
CustomerID,string,null
Country,string,null
,,
# Delta Statistics Columns,,


## Step 10 — Inspect Table Schema with DESCRIBE EXTENDED 🔍

**What this cell does:** Runs `DESCRIBE EXTENDED` to see the full table metadata — columns, types, and storage details.

**Key findings from the result:**

| Column | Inferred Type | Problem |
| --- | --- | --- |
| `InvoiceNo` | string | ✅ Correct |
| `StockCode` | string | ✅ Correct |
| `Description` | string | ✅ Correct |
| `Quantity` | string | ⚠️ Should be numeric (int/double) |
| `InvoiceDate` | string | ⚠️ Should be timestamp |
| `UnitPrice` | string | ⚠️ Should be numeric (double) |
| `CustomerID` | string | ⚠️ Should be int |
| `Country` | string | ✅ Correct |

**Why this matters:**
- Because we used a **placeholder table** (no explicit schema), `COPY INTO` inferred **every column as `STRING`**.
- This means we can't do math (`SUM(Quantity)`) or date filtering (`WHERE InvoiceDate > '2010-12-01'`) without casting — which is slow and error-prone.
- **This is the #1 downside of schema inference with CSV files.** In Steps 11-12, we'll fix this by creating a table with an **explicit schema** and using a `SELECT` transform.

**Other metadata revealed:**
- **Table type:** MANAGED (Databricks controls storage)
- **Provider:** Delta (columnar format with ACID transactions)
- **Statistics:** 51,678 bytes, 5,217 rows
- **Owner:** your user account

In [0]:
-- CREATE NEW TABLE WITH ONLY 3 COLUMNS INVOICE_CP_ALT AND CUSTOM COLUMN _INSERT_DATE

CREATE TABLE dev.bronze.invoice_cp_alt (
InvoiceNo String,
StockCode String,
Quantity double,
_insert_date TIMESTAMP
);

## Step 11 — Create a Table with Explicit Schema 🎯

**What this cell does:** Creates a new Delta table `dev.bronze.invoice_cp_alt` with **only 4 columns** and **explicit data types**:

| Column | Type | Purpose |
| --- | --- | --- |
| `InvoiceNo` | STRING | Invoice number identifier |
| `StockCode` | STRING | Product stock code |
| `Quantity` | DOUBLE | Numeric quantity (proper type!) |
| `_insert_date` | TIMESTAMP | Custom column — records when the row was loaded |

**Why this is important:**
- **Solves the schema problem** from Step 10: `Quantity` is now `DOUBLE` instead of `STRING`, enabling mathematical aggregations.
- **Selects only needed columns**: We drop `Description`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country` — only keeping what's relevant for our use case.
- **Adds a custom column** `_insert_date`: This is a common **data engineering pattern** — tracking when data was ingested is critical for auditing, debugging, and SCD (Slowly Changing Dimensions) logic.

> 🔑 Key insight: When you define an explicit schema, `COPY INTO` will **map source columns by name** to target columns. You can also transform data during the load using a `SELECT` — which we'll do in the next step.

In [0]:
-- USE COPY INTO TO LOAD DATA INTO PLACE HOLDER TABLE
COPY INTO dev.bronze.invoice_cp_alt
FROM (
    SELECT InvoiceNo, StockCode, CAST(Quantity AS DOUBLE) Quantity, current_timestamp() as _insert_date
FROM '/Volumes/dev/bronze/landing/input')
FILEFORMAT = CSV
PATTERN = '*.csv'

FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5217,5217,0


## Step 12 — COPY INTO with SELECT Transform 🔄

**What this cell does:** Uses `COPY INTO` with an inline `SELECT` to **transform data during load**.

**Breaking down the new pattern:**

```sql
COPY INTO target_table
FROM (
    SELECT col1, CAST(col2 AS DOUBLE), current_timestamp()
    FROM '/path/to/files'
)
FILEFORMAT = CSV ...
```

**Why this is powerful:**

| Feature | Benefit |
| --- | --- |
| `SELECT InvoiceNo, StockCode` | Pick only the columns you need — drop the rest |
| `CAST(Quantity AS DOUBLE)` | Convert `Quantity` from STRING to DOUBLE during load — no post-processing needed |
| `current_timestamp() as _insert_date` | Inject a **custom column** with the current load time — great for auditing |
| `FROM '...'` inside SELECT | Read directly from the file path within the query — no temp views needed |

**Result:** `5,217 rows` loaded — same as Step 6, but now with proper types and a custom audit column.

> 💡 This is the **recommended pattern** for `COPY INTO`: always use a `SELECT` to control column selection, data types, and add metadata columns during ingestion.

In [0]:
-- incremental loaded
COPY INTO dev.bronze.invoice_cp_alt
FROM (
    SELECT InvoiceNo, StockCode, CAST(Quantity AS DOUBLE) Quantity, current_timestamp() as _insert_date
FROM '/Volumes/dev/bronze/landing/input')
FILEFORMAT = CSV
PATTERN = '*.csv'

FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'header' = 'true'
)

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
2202,2202,0


## Step 13 — Incremental Load (New File Picked Up) 📈

**What this cell does:** Runs the **same** `COPY INTO` command from Step 12 again — but this time, a **second CSV file** (`2010-12-03.csv`) was added to the input directory in Step 4.

**Result:** `2,202 rows` affected, `2,202 rows` inserted, `0 corrupt files`.

**Why this is the most important cell in the notebook:**

This demonstrates the full **incremental loading** workflow:

1. ✅ Step 12: First `COPY INTO` loaded `2010-12-01.csv` → 5,217 rows
2. ✅ Step 4: New file `2010-12-03.csv` was added to the input directory
3. ✅ Step 13 (this cell): Re-ran `COPY INTO` → only **2,202 new rows** from the new file were loaded

**COPY INTO knew to skip** the first file (already loaded in Step 12) and only process the new file. This is **idempotent incremental loading** in action!

> 🎯 This is exactly how a production ingestion pipeline works:
> - Files arrive in a landing zone on a schedule
> - `COPY INTO` runs periodically (e.g., every hour via a Databricks Job)
> - Only new files are loaded — no duplicates, no manual tracking
> - The checkpoint is managed automatically by the Delta table

In [0]:
select count(1) from dev.bronze.invoice_cp_alt

count(1)
7419


## Step 14 — Final Row Count Verification ✅

**What this cell does:** Counts all rows in `dev.bronze.invoice_cp_alt`.

**Result:** `7,419` rows total.

**The math checks out:**

| Load Step | File | Rows Loaded | Running Total |
| --- | --- | --- | --- |
| Step 12 (first load) | `2010-12-01.csv` | 5,217 | 5,217 |
| Step 13 (incremental) | `2010-12-03.csv` | 2,202 | **7,419** |

**Why this matters:**
- 5,217 + 2,202 = 7,419 ✅ — the counts match perfectly.
- This confirms that `COPY INTO` loaded **both** files over two runs, with **no duplicates** and **no data loss**.
- This final verification closes the loop on our entire ingestion demo.

---

# 🏆 Summary — COPY INTO Deep Dive

## What We Built

This notebook walked through a **complete data ingestion pipeline** using `COPY INTO` — from setting up storage to incremental loading with schema control.

## Journey at a Glance

```
Volume Setup  →  Stage Files  →  Placeholder Table  →  First Load (5,217 rows)
     │                                              │
     │                                    Idempotency Test (0 rows) ✅
     │                                              │
     │                                    Schema Inspection (all STRINGs ⚠️)
     │                                              │
     │                                    Explicit Schema Table 🎯
     │                                              │
     │                                    COPY INTO + SELECT Transform (5,217 rows)
     │                                              │
     └──── New file added ──→ Incremental Load (2,202 rows) 📈
                                    │
                                    Final Count = 7,419 ✅
```

## Key Takeaways

| # | Lesson | Why It Matters |
| --- | --- | --- |
| 1 | **COPY INTO is idempotent** | Re-running won't duplicate data — safe for scheduled jobs |
| 2 | **Schema inference defaults to STRING** | Always define explicit schema for CSV loads when types matter |
| 3 | **Use SELECT to transform during load** | Cast types, select columns, add audit columns — all in one step |
| 4 | **COPY INTO supports incremental loading** | New files in the source path are automatically picked up |
| 5 | **Always verify with count + describe** | Don't assume — verify row counts and schema after every load |
| 6 | **Volumes provide governed storage** | Use Unity Catalog volumes as your landing zone for files |

## COPY INTO vs Auto Loader — When to Use Which?

| Criteria | COPY INTO | Auto Loader |
| --- | --- | --- |
| **Complexity** | Simple (SQL only) | More setup (Python/Scala) |
| **File volume** | Small to medium | Large to very large |
| **Schema evolution** | Basic (`mergeSchema`) | Advanced (schema inference + evolution modes) |
| **Processing mode** | Batch (triggered) | Streaming (continuous or triggered) |
| **Best for** | Simple batch loads, quick prototypes | Production pipelines, high-throughput ingestion |

## Production Checklist for COPY INTO

- ✅ Define an **explicit table schema** — don't rely on inference
- ✅ Use a **SELECT** to control column mapping and types
- ✅ Add a `_insert_date` or `_loaded_at` **audit column**
- ✅ **Verify** row counts after every load
- ✅ Schedule with a **Databricks Job** for automated incremental loading
- ✅ Use **Unity Catalog volumes** for governed file storage
- ✅ Clean up **duplicate cells** and add documentation (like this notebook!)

---

> 📚 **Further Reading:**
> - [COPY INTO documentation](https://docs.databricks.com/sql/language-manual/delta-copy-into)
> - [Unity Catalog volumes](https://docs.databricks.com/en/catalogs/volumes.html)
> - [Medallion architecture](https://www.databricks.com/glossary/medallion-architecture)

*This notebook is part of an Interview Prep series covering essential Databricks data engineering concepts.*